In [ ]:
import pandas as pd
from sqlalchemy import text

from literature_ai.db import ENGINE
from literature_ai.search_service.search.vector_search import vector_search_async
from literature_ai.search_service.search.keyword_search import keyword_search
from literature_ai.search_service.search.chunk_search import rag_paper_chunks, CHUNKS_TABLE
from literature_ai.search_service.data_collect.collect_full_papers import process_papers_by_id, FULLTEXT_TABLE

QUERY = "ML for optical coherence tomography diagnosis"

In [5]:
vector_results = await vector_search_async(QUERY, run_id=1, n_results=20)
display(pd.DataFrame(vector_results))

/Users/adam/Documents/Projects/Literature-Review-AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/adam/Documents/Projects/Literature-Review-AI/.venv/lib/python3.13/site-packages/filelock/_soft_rw/_async.py:10: RuntimeWarning: coroutine 'vector_search_async' was never awaited
  from ._sync import SoftReadWriteLock
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 92691.80it/s]
There are adapters available but none are activated for the forward pass.


,paperId,title,abstract,year,venue,citationCount,url,DOI,distance
0,b954efe5e46b8952f5a8daf42e7e535119b5408b,Development and Validation of a Deep Learning ...,None,2017,Journal of the American Medical Association (J...,2063,https://jamanetwork.com/journals/jama/articlep...,10.1001/jama.2017.18152,0.217084
1,5c45a5d05ac564adb67811eeb9d41d6460c70135,Development and Validation of a Deep Learning ...,None,2016,Journal of the American Medical Association (J...,6883,https://jamanetwork.com/journals/jama/articlep...,10.1001/jama.2016.17216,0.223362
2,ba913e2c03ece1c75f0af4d16dd11c7ffbc6e3ba,Diagnostic Assessment of Deep Learning Algorit...,None,2017,Journal of the American Medical Association (J...,3158,https://jamanetwork.com/journals/jama/articlep...,10.1001/jama.2017.14585,0.225067
3,dc0c84b7c5e6521216da789f8171544709120cf0,Opportunities and obstacles for deep learning ...,"Deep learning, which describes a class of mach...",2017,bioRxiv,2020,https://www.biorxiv.org/content/biorxiv/early/...,10.1098/rsif.2017.0387,0.227537
4,9406246f6972c03e5bfaac4df4676648dc4ac935,Deep Learning Techniques for Automatic MRI Car...,"Delineation of the left ventricular cavity, my...",2018,IEEE Transactions on Medical Imaging,2276,https://qmro.qmul.ac.uk/xmlui/bitstream/123456...,10.1109/TMI.2018.2837502,0.229409
5,89a816719613e220a64ab2590c938c23bbfe187e,CheXpert: A Large Chest Radiograph Dataset wit...,"Large, labeled datasets have driven deep learn...",2019,AAAI Conference on Artificial Intelligence,3583,https://aaai.org/ojs/index.php/AAAI/article/do...,10.1609/aaai.v33i01.3301590,0.234148
6,7ae2783a9196fb4bc2a610ae812d19722daddce5,Applications of machine learning to machine fa...,Abstract Intelligent fault diagnosis (IFD) ref...,2020,,2561,http://bura.brunel.ac.uk/bitstream/2438/20040/...,10.1016/j.ymssp.2019.106587,0.234960
7,6ff909c6fe089fc8ebfc64eca0f0c3cc34ba277f,A survey on deep learning in medical image ana...,"Deep learning algorithms, in particular convol...",2017,Medical Image Anal.,13760,https://ars.els-cdn.com/content/image/1-s2.0-S...,10.1016/j.media.2017.07.005,0.237867
8,8df72c48a7ce4418c683c4dd9bb300558ac71d47,"Deep learning for healthcare: review, opportun...",None,2018,Briefings Bioinform.,2627,https://europepmc.org/articles/pmc6455466?pdf=...,10.1093/bib/bbx044,0.240149
9,7ab0f0da686cd4094fd96f5a30e0b6072525fd09,Deep Learning in Medical Image Analysis,The computer-assisted analysis for better inte...,2017,Annual Review of Biomedical Engineering,2924,https://www.annualreviews.org/doi/pdf/10.1146/...,10.1146/annurev-bioeng-071516-044442,0.242201


In [6]:
keyword_results = keyword_search(QUERY, n_results=20)
display(pd.DataFrame(keyword_results))

""


In [7]:
paper_ids = [r["paperId"] for r in vector_results]
metrics = process_papers_by_id(paper_ids)

2026-09-04 17:39:57.441 | WARNING  | literature_ai.search_service.data_collect.collect_full_papers:download_pdf:73 - Failed to download https://jamanetwork.com/journals/jama/articlepdf/2665775/jama_ting_2017_oi_170140.pdf: 403 Client Error: Forbidden for url: https://jamanetwork.com/journals/jama/articlepdf/2665775/jama_ting_2017_oi_170140.pdf
2026-09-04 17:39:57.489 | WARNING  | literature_ai.search_service.data_collect.collect_full_papers:download_pdf:73 - Failed to download https://jamanetwork.com/journals/jama/articlepdf/2588763/joi160132.pdf: 403 Client Error: Forbidden for url: https://jamanetwork.com/journals/jama/articlepdf/2588763/joi160132.pdf
2026-09-04 17:39:57.532 | WARNING  | literature_ai.search_service.data_collect.collect_full_papers:download_pdf:73 - Failed to download https://jamanetwork.com/journals/jama/articlepdf/2665774/jama_ehteshami_bejnordi_2017_oi_170113.pdf: 403 Client Error: Forbidden for url: https://jamanetwork.com/journals/jama/articlepdf/2665774/jama_eh

KeyboardInterrupt: 

In [ ]:
print(metrics.inserted)

In [ ]:
with ENGINE.connect() as conn:
    rows = conn.execute(
        text('SELECT "paperId" FROM {} WHERE "paperId" = ANY(:ids) AND "status" = \'success\''.format(FULLTEXT_TABLE)),
        {"ids": paper_ids},
    ).fetchall()
successful_ids = [r[0] for r in rows]

for paper_id in successful_ids:
    with ENGINE.connect() as conn:
        rows = conn.execute(
            text('SELECT section_header FROM {} WHERE "paperId" = :id ORDER BY section_index'.format(CHUNKS_TABLE)),
            {"id": paper_id},
        ).fetchall()
    headers = list(dict.fromkeys(r[0] for r in rows))
    print(paper_id, headers)

In [ ]:
rag_results = rag_paper_chunks(successful_ids, QUERY, n_results=10)
display(pd.DataFrame(rag_results["results"]))